# 00 — Лаборатория основ: цена, разложение премии и доска

Эта лаборатория делает урок осязаемым. Вы:

1. Оцените опционы DEMO по страйкам и срокам через `pricing.bsm_price`.
2. **Разложите** каждую премию на внутреннюю и временную стоимость и увидите пропорцию.
3. Изучите настоящую опционную доску `DEMO`: bid/ask, спреды, ликвидность, денежность.

Всё работает оффлайн на вложенных тестовых досках. Спот DEMO — **$100**, IV ~**25%**.

## Подготовка

Время в этой библиотеке всегда в **годах**: 45 DTE — это `45/365`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import pricing, data

SPOT = 100.0      # базовый актив DEMO
VOL  = 0.25       # подразумеваемая волатильность ~25%
DTE  = 45
t    = DTE / 365  # конвенция: время в ГОДАХ
round(t, 4)

## 1. Оценим один опцион

Оценим ATM-колл 100 на 45 DTE. Ниже сравним его с котировкой mid из доски.

In [ ]:
atm_call = pricing.bsm_price('call', SPOT, strike=100, t=t, vol=VOL)
atm_put  = pricing.bsm_price('put',  SPOT, strike=100, t=t, vol=VOL)
print(f'ATM колл 100: {atm_call:.2f}')
print(f'ATM пут  100: {atm_put:.2f}')

## 2. Оценим колл по всем страйкам

Пройдём страйки от глубокого ITM (80) до глубокого OTM (120). Смотрите, как цена плавно падает.

In [ ]:
strikes = np.arange(80, 122.5, 2.5)
call_px = [pricing.bsm_price('call', SPOT, k, t, VOL) for k in strikes]
for k, p in zip(strikes, call_px):
    print(f'страйк {k:6.1f}  колл {p:6.2f}')

## 3. Разложим премию на внутреннюю и временную стоимость

Ключевой навык из урока: `премия = внутренняя + временная`.
Внутренняя стоимость колла — это `max(spot - strike, 0)`; временная — всё, что осталось.

In [ ]:
def decompose_call(spot, strike, price):
    intrinsic = max(spot - strike, 0.0)
    extrinsic = price - intrinsic
    return intrinsic, extrinsic

for k in [80, 90, 100, 110, 120]:
    p = pricing.bsm_price('call', SPOT, k, t, VOL)
    intr, extr = decompose_call(SPOT, k, p)
    print(f'страйк {k:5.1f}  цена {p:6.2f}  внутренняя {intr:6.2f}  временная {extr:6.2f}')

Обратите внимание: **у ATM-колла 100 больше всего временной стоимости** — рынок берёт максимум за
неопределённость ровно там, где исход наиболее сомнителен. Посмотрим на это картинкой.

In [ ]:
prices    = np.array([pricing.bsm_price('call', SPOT, k, t, VOL) for k in strikes])
intrinsic = np.maximum(SPOT - strikes, 0.0)
extrinsic = prices - intrinsic
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(strikes, intrinsic, width=1.6, label='внутренняя')
ax.bar(strikes, extrinsic, width=1.6, bottom=intrinsic, label='временная')
ax.axvline(SPOT, color='k', ls='--', lw=1, label='спот=100')
ax.set_xlabel('страйк'); ax.set_ylabel('премия колла'); ax.legend(); ax.set_title('Колл DEMO 45 DTE: внутренняя и временная стоимость')
plt.show()

## 4. Временная стоимость и часы распада

Зафиксируем спот и страйк на деньгах и будем сокращать время. Временная стоимость (а у ATM это вся
премия) уменьшается — и уменьшается *тем быстрее*, чем ближе экспирация.

In [ ]:
for dte in [90, 60, 45, 30, 21, 7, 1]:
    p = pricing.bsm_price('call', SPOT, 100, dte/365, VOL)
    print(f'{dte:3d} DTE   ATM колл {p:5.2f}   (вся — временная)')

## 5. Загрузим и прочитаем настоящую доску DEMO

`data.load_sample_chain` возвращает DataFrame с документированными колонками. Осмотрим его.

In [ ]:
chain = data.load_sample_chain('DEMO')
print('доступные доски:', data.list_sample_chains())
print('спот:', chain['spot'].iloc[0])
chain.head()

## 6. Вырежем коллы на 45 DTE и посмотрим на ликвидность

Отфильтруем коллы на 45 DTE, посчитаем спред bid-ask и увидим, как рынок **разрежается в
крыльях**.

In [ ]:
c45 = chain[(chain.kind == 'call') & (chain.expiry_days == 45)].copy()
c45['spread'] = c45['ask'] - c45['bid']
c45['spread_pct'] = 100 * c45['spread'] / c45['mid'].where(c45['mid'] > 0)
cols = ['strike', 'bid', 'ask', 'mid', 'spread', 'spread_pct', 'open_interest', 'volume']
c45[cols].round(2).to_string(index=False)

Колл 100: спред шириной в цент, OI > 8 000 — глубоко торгуемо. Дальний OTM-колл 130: спред шириной
во всю стоимость опциона, тонкий OI — ловушка. **Ликвидность — тот фильтр, который применяют первым.**

## 7. Модель против рынка

Сравним нашу цену BSM с плоской волатильностью и котировку mid из доски на каждом страйке. Они
расходятся, потому что рынок закладывает *свою IV в каждый страйк* (скью — модуль 02). Колонка `iv`
в доске это показывает.

In [ ]:
c45 = c45.sort_values('strike')
c45['model_flatvol'] = [pricing.bsm_price('call', SPOT, k, 45/365, 0.25) for k in c45.strike]
c45[['strike', 'mid', 'model_flatvol', 'iv']].round(3).to_string(index=False)

## Эксперименты

Меняйте это и перезапускайте — именно здесь интуиция и закрепляется:

1. В разделе 3 замените `'call'` на `'put'` и перепишите `decompose_call` под путы
   (`intrinsic = max(strike - spot, 0)`). У какого страйка пута теперь больше всего временной
   стоимости?
2. В разделе 4 поднимите `VOL` с 0.25 до 0.45 и перезапустите таблицу распада. Насколько больше
   временной стоимости приходится распадать при высокой IV?
3. В разделе 6 поменяйте `expiry_days == 45` на `== 7` и `== 180`. Как меняется ликвидность в
   крыльях (OI, спреды) с изменением DTE?
4. Оцените колл 90 на 45 DTE, затем на 7 DTE. Какая доля его стоимости внутренняя в каждом случае
   и почему глубокий ITM-опцион почти не распадается?
5. Загрузите доску `HIGHVOL` (спот 62, IV ~55%) и повторите раздел 6. Спреды ATM у неё уже или шире
   в процентном выражении, чем у DEMO?